# 10 - Robustness and Sensitivity Analysis

**Purpose:** Assess sensitivity of inflation findings to forecast horizon, window size, model family, and dataset subset choices.

**Inputs:** results/metrics/ (notebooks 06-07); results/tables/ (notebook 08).

**Outputs:** Sensitivity tables/figures under results/tables/ and results/figures/.

**Runtime estimate:** 30-90 minutes (reruns parts of the grid under varied settings).

## Horizon sensitivity

In [1]:
import pandas as pd

from sdg_leakage_audit.utils.paths import data_processed_dir, results_tables_dir, to_project_relative

inflation_df = pd.read_csv(results_tables_dir() / "inflation_by_cell.csv")

# Every protocol in notebooks 06-07 was already run across each dataset's own configured
# forecast horizons, so horizon sensitivity can be read directly off the existing results --
# no re-run needed.
horizon_sensitivity = (
    inflation_df[inflation_df["protocol_id"] != "P0"]
    .groupby(["protocol_id", "horizon"])["mae_inflation_pct"]
    .mean()
    .unstack("horizon")
    .round(2)
)
out_path = results_tables_dir() / "robustness_horizon_sensitivity.csv"
horizon_sensitivity.to_csv(out_path)
print(f"Saved -> {to_project_relative(out_path)}")
horizon_sensitivity


Saved -> results\tables\robustness_horizon_sensitivity.csv


horizon,1,6,24,60,144
protocol_id,,,,,
P1,-4.59,-8.81,12.38,15.42,6.09
P2,-0.20,-0.39,-4.25,-3.18,1.25
P3,0.05,-2.41,8.84,-4.74,-1.44
P4,-2.40,-0.13,11.08,0.81,-0.04
P5,5.13,0.78,2.22,-5.00,-7.93
P6,-4.52,-3.37,17.55,14.17,5.83


## Window-size sensitivity

In [2]:
from sdg_leakage_audit.evaluation.runner import run_single_cell_on_frame
from sdg_leakage_audit.features.build import build_feature_matrix
from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_interim_dir

datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]
WINDOW_DATASET = "household_power_consumption"
WINDOW_MODEL = "ridge"
cfg = datasets_cfg[WINDOW_DATASET]
cleaned_df = pd.read_parquet(data_interim_dir() / f"{WINDOW_DATASET}.parquet")

# Compare the default window set (this dataset's configured forecast_horizons, as used in
# notebook 04) against a narrower and a wider alternative, holding everything else fixed.
window_configs = {
    "narrow": [1],
    "default": cfg["forecast_horizons"],
    "wide": cfg["forecast_horizons"] + [120, 240],
}

FAST_CONFIG = {"data_subsample_frac": 0.05, "model_param_grid": "params_fast"}
window_rows = []
for window_label, windows in window_configs.items():
    features_df = build_feature_matrix(
        cleaned_df, target_column=cfg["target_column"], lags=windows, rolling_windows=windows
    )
    for protocol_id in ["P0", "P1"]:
        result = run_single_cell_on_frame(
            features_df, f"{WINDOW_DATASET}_{window_label}", cfg["target_column"], cfg["season_length"],
            protocol_id, WINDOW_MODEL, horizon=1, seed=0, config=FAST_CONFIG,
        )
        window_rows.append({"window_config": window_label, "n_windows": len(windows), **result})

window_df = pd.DataFrame(window_rows)
window_pivot = window_df.pivot_table(index="window_config", columns="protocol_id", values="mae")
window_pivot["mae_inflation_pct"] = (window_pivot["P0"] - window_pivot["P1"]) / window_pivot["P0"] * 100
out_path = results_tables_dir() / "robustness_window_size_sensitivity.csv"
window_pivot.to_csv(out_path)
print(f"Saved -> {to_project_relative(out_path)}")
window_pivot.round(3)


2026-10-21 19:34:27 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:30 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:36 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:40 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:47 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:51 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


Saved -> results\tables\robustness_window_size_sensitivity.csv


protocol_id,P0,P1,mae_inflation_pct
window_config,,,
default,0.087,0.083,4.642
narrow,0.087,0.082,5.697
wide,0.086,0.085,1.087


## Model-family sensitivity

In [3]:
from sdg_leakage_audit.models.factory import get_model_family

inflation_df["model_family"] = inflation_df["model_name"].apply(get_model_family)
family_sensitivity = (
    inflation_df[inflation_df["protocol_id"] != "P0"]
    .groupby(["protocol_id", "model_family"])["mae_inflation_pct"]
    .mean()
    .unstack("model_family")
    .round(2)
)
out_path = results_tables_dir() / "robustness_model_family_sensitivity.csv"
family_sensitivity.to_csv(out_path)
print(f"Saved -> {to_project_relative(out_path)}")
family_sensitivity


Saved -> results\tables\robustness_model_family_sensitivity.csv


model_family,baseline,linear,neural,tree_ensemble
protocol_id,,,,
P1,-20.62,24.72,-16.47,36.39
P2,-0.05,0.44,-3.28,-0.97
P3,1.16,3.46,0.40,1.80
P4,-5.89,6.49,-0.13,11.54
P5,-0.05,1.22,0.72,4.95
P6,-15.02,26.01,-14.61,38.36


## Dataset-subset sensitivity

In [4]:
beijing_df = pd.read_parquet(data_processed_dir() / "beijing_multisite_air_quality_features.parquet")
beijing_cfg = datasets_cfg["beijing_multisite_air_quality"]

stations = sorted(beijing_df["station"].unique())[:3]  # a representative subset, not all 12
print(f"Checking station sensitivity for: {stations}")

station_rows = []
for station in stations:
    station_df = beijing_df.loc[beijing_df["station"] == station].drop(columns=["station"])
    for protocol_id in ["P0", "P1"]:
        result = run_single_cell_on_frame(
            station_df, f"beijing_{station}", beijing_cfg["target_column"], beijing_cfg["season_length"],
            protocol_id, "ridge", horizon=6, seed=0, config=FAST_CONFIG,
        )
        station_rows.append({"station": station, **result})

station_df_results = pd.DataFrame(station_rows)
station_pivot = station_df_results.pivot_table(index="station", columns="protocol_id", values="mae")
station_pivot["mae_inflation_pct"] = (station_pivot["P0"] - station_pivot["P1"]) / station_pivot["P0"] * 100
out_path = results_tables_dir() / "robustness_dataset_subset_sensitivity.csv"
station_pivot.to_csv(out_path)
print(f"Saved -> {to_project_relative(out_path)}")
print(
    "\nNote: notebooks 03-09 all used only the alphabetically-first station as Beijing's "
    "representative series -- this check is exactly what tests whether that choice happened "
    "to be unusually (non-)leaky relative to other stations."
)
station_pivot.round(3)


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


Checking station sensitivity for: ['Aotizhongxin', 'Changping', 'Dingling']


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-21 19:34:55 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


Saved -> results\tables\robustness_dataset_subset_sensitivity.csv

Note: notebooks 03-09 all used only the alphabetically-first station as Beijing's representative series -- this check is exactly what tests whether that choice happened to be unusually (non-)leaky relative to other stations.


protocol_id,P0,P1,mae_inflation_pct
station,,,
Aotizhongxin,45.352,45.281,0.156
Changping,35.468,42.001,-18.418
Dingling,36.768,33.974,7.599


## Findings / what to carry forward

- **Horizon sensitivity** (read directly off existing notebook 06-07 results, no re-run): see
  `results/tables/robustness_horizon_sensitivity.csv` -- whether a protocol's inflation grows,
  shrinks, or stays flat as forecast horizon increases, per protocol.
- **Window-size sensitivity** (household_power_consumption, Ridge): comparing narrow ([1]),
  default (this dataset's configured horizons), and wide (+120, +240 lag/rolling windows)
  feature sets shows whether P1's inflation is an artifact of a specific window choice or
  persists across window sizes. See `results/tables/robustness_window_size_sensitivity.csv`.
- **Model-family sensitivity**: inflation aggregated by family (baseline/linear/tree_ensemble/
  neural) rather than individual model, in `results/tables/robustness_model_family_sensitivity.csv`
  -- answers whether leakage effects are a tree-ensemble-specific artifact, a neural-network
  artifact, or genuinely cross-family.
- **Dataset-subset sensitivity** (3 of Beijing's 12 stations): every earlier notebook (03-09)
  reduced Beijing to one representative station for tractability -- this check tests whether
  that single station happened to be unusually (non-)leaky compared to its peers. See
  `results/tables/robustness_dataset_subset_sensitivity.csv`.
- All four checks reuse `evaluation.runner.run_single_cell_on_frame` (the dataset-id-free core
  of `run_single_cell`, factored out specifically for this notebook) rather than duplicating
  split/fit/metric logic inline -- consistent with keeping business logic in `src`.
- Sample-size caveat carried forward again: every check here still runs under the `fast`
  profile's single seed and small subsample/hyperparameter grid, so these are sensitivity
  DIRECTIONS worth re-checking at `full`-profile scale, not final robustness conclusions.
- Next (notebook 11): assemble final tables and figures from the results produced across
  notebooks 06-10, with consistent styling and captions.
